In [ ]:
!pip install wikipedia-api

import wikipediaapi

In [ ]:
!pip install -U transformers

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-1.7B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re


tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")


def generate_bio(prompt, max_new_tokens=1024):
    messages = [{"role": "user", "content": prompt + " /no_think"}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()


In [6]:
# Getting wiki page
def get_wiki_context(entity_name, lang="en", sentences=5):
    wiki = wikipediaapi.Wikipedia(language=lang, user_agent="research-bot/1.0")
    page = wiki.page(entity_name)
    if not page.exists():
        print(f"[WARN] No Wikipedia page found for '{entity_name}' in lang='{lang}'")
        return None
    return ". ".join(page.summary.split(". ")[:sentences])

# RAG generate
def generate_bio_with_rag(prompt, entity_name, lang="en", **kwargs):
    context = get_wiki_context(entity_name, lang=lang)
    if context:
        augmented_prompt = f"Use the following factual context:\n{context}\n\n{prompt}"
    else:
        augmented_prompt = prompt

    print("=" * 60)
    print("AUGMENTED PROMPT:")
    print("=" * 60)
    print(augmented_prompt)
    print("=" * 60 + "\n")

    return generate_bio(augmented_prompt, **kwargs)

In [ ]:
# English RAG generation
en_bio_rag = generate_bio_with_rag(
    "Write a biography of Xi Jinping.",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nENGLISH (RAG - English Wiki):\n", en_bio_rag)

In [ ]:
# Bengali w/ English RAG
bn_bio_rag_en = generate_bio_with_rag(
    "শি জিনপিংয়ের জীবনী লিখুন।",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nBENGALI (RAG - English Wiki):\n", bn_bio_rag_en)

# Bengali w/ Native RAG
bn_bio_rag_native = generate_bio_with_rag(
    "শি জিনপিংয়ের জীবনী লিখুন।",
    entity_name="শি জিনপিং",
    lang="bn"
)
print("\nBENGALI (RAG - Bengali Wiki):\n", bn_bio_rag_native)

In [ ]:
# Hindi w/ English RAG
hi_bio_rag_en = generate_bio_with_rag(
    "शी जिनपिंग की जीवनी लिखें।",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nHINDI (RAG - English Wiki):\n", hi_bio_rag_en)

# Hindi w/ Native RAG
hi_bio_rag_native = generate_bio_with_rag(
    "शी जिनपिंग की जीवनी लिखें।",
    entity_name="शी जिनपिंग",
    lang="hi"
)
print("\nHINDI (RAG - Hindi Wiki):\n", hi_bio_rag_native)

In [ ]:
# Swahili w/ English RAG
sw_bio_rag_en = generate_bio_with_rag(
    "Andika wasifu wa Xi Jinping.",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nSWAHILI (RAG - English Wiki):\n", sw_bio_rag_en)

# Swahili w/ Native RAG
sw_bio_rag_native = generate_bio_with_rag(
    "Andika wasifu wa Xi Jinping.",
    entity_name="Xi Jinping",
    lang="sw"
)
print("\nSWAHILI (RAG - Swahili Wiki):\n", sw_bio_rag_native)

In [ ]:
# English - no retrieval
en_bio = generate_bio("Write a biography of Xi Jinping.")
print("ENGLISH:\n", en_bio)

In [ ]:
# Bengali - no retrieval
bn_bio = generate_bio("শি জিনপিং-এর জীবনী লিখুন।")
# "Write a biography of Xi Jinping."
print("\nBENGALI:\n", bn_bio)

In [ ]:
# Swahili - no retrieval
sw_bio = generate_bio("Andika wasifu wa Xi Jinping.")
# "Write a biography of Xi Jinping."
print("\nSWAHILI:\n", sw_bio)

In [ ]:
# Hindi - no retrieval
hi_bio = generate_bio("शी जिनपिंग की जीवनी लिखें।")
# "Write a biography of Xi Jinping."
print("\nHINDI:\n", hi_bio)

In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [ ]:
# import os
# from openai import OpenAI

# client = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key=os.environ["HF_TOKEN"],
# )

# completion = client.chat.completions.create(
#     model="Qwen/Qwen3-1.7B",
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )

# print(completion.choices[0].message)